# Full strategy evaluation and P&L attribution

I use fixed seeds here so the numerical examples remain comparable when the implementation changes.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/"src"))
pd.set_option("display.max_columns", 30)

I bring the separate pieces together here: paired strategy comparisons, exact P&L components, bootstrap intervals, and the validation residuals.

In [2]:
summary=pd.read_csv(ROOT/"outputs/tables/strategy_summary.csv")
summary[["scenario","strategy","mean_terminal_pnl","pnl_5_percentile","mean_absolute_inventory","mean_markout_per_unit"]].head(15)

,scenario,strategy,mean_terminal_pnl,pnl_5_percentile,mean_absolute_inventory,mean_markout_per_unit
0,clean,fixed,3.412374,-5.770058,4.6467,0.055083
1,clean,full_adaptive,2.916983,0.395857,2.0654,0.090538
2,clean,full_risk,2.916983,0.395857,2.0654,0.090538
3,clean,inventory,3.497412,0.231789,2.2611,0.053055
4,clean,inventory_volatility,3.123969,-0.015486,2.2707,0.081872
5,high_volatility,fixed,3.083607,-21.220302,4.7217,0.057060
6,high_volatility,full_adaptive,0.278773,-6.298733,1.6645,0.151353
7,high_volatility,full_risk,-0.408777,-8.574559,1.5193,0.151204
8,high_volatility,inventory,1.513807,-8.090779,2.1998,0.050633
9,high_volatility,inventory_volatility,0.922392,-5.224339,1.7390,0.133345


In [3]:
attrib=pd.read_csv(ROOT/"outputs/tables/pnl_attribution_summary.csv")
attrib[attrib.scenario=="stress"]

,scenario,strategy,total_spread_capture,maker_fee_pnl,information_price_pnl,noise_price_pnl,jump_price_pnl,forced_execution_pnl,terminal_adjustment_pnl,terminal_pnl
20,stress,fixed,6.841297,-0.124675,-38.910349,4.447332,-31.1625,0.00000,-3.20932,-62.118215
21,stress,full_adaptive,2.018004,-0.010050,-1.521274,0.110642,-2.1150,0.00000,-0.15372,-1.671398
22,stress,full_risk,1.817933,-0.009250,-1.001085,0.284273,-2.0925,-0.04712,-0.10620,-1.153948
23,stress,inventory,8.251581,-0.151025,-32.241314,-0.553897,-9.2025,0.00000,-0.81856,-34.715715
24,stress,inventory_volatility,3.654088,-0.021750,-6.298689,0.296550,-4.2525,0.00000,-0.43772,-7.060020


In [4]:
pairs=pd.read_csv(ROOT/"outputs/tables/paired_strategy_comparisons.csv")
pairs[(pairs.scenario.isin(["toxic","stress"])) & (pairs.strategy_a.isin(["inventory_volatility","full_adaptive","full_risk"]))]

,scenario,strategy_a,strategy_b,mean_difference,lower_bound,upper_bound,probability_positive
9,toxic,inventory_volatility,inventory,2.190395,0.367254,4.149605,0.9912
10,toxic,full_adaptive,inventory_volatility,0.677918,-0.714454,1.910263,0.8352
11,toxic,full_risk,full_adaptive,0.086692,0.000000,0.260076,0.6464
21,stress,inventory_volatility,inventory,27.655695,19.343325,37.168388,1.0000
22,stress,full_adaptive,inventory_volatility,5.388622,0.441899,10.868005,0.9836
23,stress,full_risk,full_adaptive,0.517450,-1.307231,2.749582,0.6780


In [5]:
validation=pd.read_csv(ROOT/"outputs/tables/validation_summary.csv")
validation

,scenario,strategy,positive_mid_prices,non_crossed_quotes,interval_reconciliation_pass,terminal_reconciliation_pass,maximum_interval_error,terminal_reconciliation_error,markout_identity_pass,inventory_limit_pass
0,toxic,fixed,True,True,True,True,2.651213e-13,3.090861e-12,True,NaN
1,regime_switching,full_adaptive,True,True,True,True,6.628031e-14,3.304024e-13,True,NaN
2,stress,full_risk,True,True,True,True,5.684342e-14,4.263256e-14,True,True
